# Introdução ao PyTorch: do primeiro tensor à classificação de dígitos

**Introdução à Aprendizagem Automática · Tutorial de estudo autónomo**

Vamos aprender a construir, treinar e avaliar uma **Multilayer Perceptron (MLP)** no MNIST.
No final, reutilizamos o mesmo processo numa pequena **Convolutional Neural Network (CNN)**.
O objetivo é perceber o percurso completo: dados → previsões → loss → gradientes → atualização → avaliação.

**Pré-requisitos.** Python básico e os conceitos de aprendizagem supervisionada abordados na unidade curricular.
Assumimos instaladas as bibliotecas indicadas no prefácio do projeto: `numpy`, `scipy`, `matplotlib`,
`pandas` e `scikit-learn`. Aqui usamos apenas `matplotlib`, além de **`torch` e `torchvision`**, que podem precisar de instalação.
Não é necessária experiência anterior com PyTorch nem uma GPU.

**Como trabalhar.** Escolhe o kernel do teu ambiente Python e executa as células por ordem, ou usa
*Restart Kernel and Run All Cells*. Lê as interpretações e tenta responder às perguntas antes de avançar.
As sugestões de exploração são opcionais: o notebook completo corre sem alterações manuais.
Reserva cerca de 60–90 minutos para estudar; o tempo de cálculo depende do computador.
Usamos uma amostra do treino para manter a execução viável em CPU.

**Percurso:** 1. Ambiente · 2. Tensores e autograd · 3. MNIST e batches · 4. MLP e loss ·
5. Treino e validação · 6. Hiperparâmetros · 7. CNN · 8. Teste final · 9. Guardar e reutilizar.

Este tutorial usa dígitos como exemplo independente. A secção final explica o que se transfere para o projeto,
incluindo os cuidados adicionais necessários quando existem vários exemplos por paciente.

## 1. Preparar o ambiente

Se as dependências já estiverem instaladas, não é preciso instalar nada. A célula seguinte contém o comando
opcional; para o usar, remove o `#` inicial e executa-a. `%pip` instala no ambiente do kernel do notebook.
Depois de uma instalação, reinicia o kernel e volta ao início.

O comando é suficiente para o percurso em CPU. Para instalar uma versão com suporte CUDA adequado à tua GPU,
consulta o [seletor oficial do PyTorch](https://pytorch.org/get-started/locally/).
Em Jupyter, confirma que escolheste o kernel do ambiente onde instalaste as bibliotecas.

In [1]:
# Instalação opcional, apenas se necessária:
%pip install torch torchvision matplotlib

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 1.4/1.4 MB 14.6 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from pathlib import Path
from copy import deepcopy
from time import perf_counter
import math
import random
import sys

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset, random_split
import torchvision
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({"figure.figsize": (9, 3.5), "font.size": 10})

SEED = 42

def definir_seed(seed=SEED):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

definir_seed()
# Limitar o paralelismo evita sobrecarga em operações pequenas em CPU.
torch.set_num_threads(min(4, torch.get_num_threads()))
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__} | torchvision {torchvision.__version__}")
print("Dispositivo:", device)

Python 3.14.5 | PyTorch 2.15.0+cpu | torchvision 0.29.2+cpu
Dispositivo: cpu


O modelo e cada batch terão de estar no mesmo `device`; `.to(device)` faz essa transferência.
As seeds tornam a divisão dos dados, a inicialização e a ordem dos batches repetíveis no mesmo contexto.
Não garantem resultados idênticos entre versões, dispositivos ou todas as operações em GPU.

## 2. Tensores e diferenciação automática

Um **tensor** é uma coleção de números organizada por dimensões. Uma imagem MNIST tem forma
`[1, 28, 28]`: um canal de cinzentos, 28 linhas e 28 colunas. Um batch de $B$ imagens tem
forma `[B, 1, 28, 28]`. A MLP recebe cada imagem como um vetor com $28\times28=784$ valores.
Achatar a imagem mantém os valores; deixa de representar explicitamente a vizinhança entre píxeis.

In [ ]:
imagens_exemplo = torch.zeros(4, 1, 28, 28)
vetores_exemplo = imagens_exemplo.flatten(start_dim=1)
print("Batch de imagens:", tuple(imagens_exemplo.shape))
print("Batch de vetores:", tuple(vetores_exemplo.shape))
print("Tipo dos valores:", imagens_exemplo.dtype)

**Autograd** acompanha operações para calcular derivadas automaticamente. Num modelo, queremos
saber como cada peso influencia a loss. `backward()` aplica a regra da cadeia e coloca essas
derivadas em `.grad`; não altera os pesos por si só.

Antes de usar uma rede, vamos verificar $L=(w-3)^2$. A derivada é $2(w-3)$: em $w=1$, vale $-4$.
Um passo de descida do gradiente, $w\leftarrow w-\eta\,dL/dw$, deve aproximar $w$ de 3.

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
loss_exemplo = (w - 3) ** 2
loss_exemplo.backward()
print(f"w = {w.item():.1f}; loss = {loss_exemplo.item():.1f}; derivada = {w.grad.item():.1f}")

with torch.no_grad():
    w -= 0.1 * w.grad
print(f"Após um passo: w = {w.item():.1f}; nova loss = {((w - 3) ** 2).item():.2f}")

`torch.no_grad()` desativa o registo de operações para autograd nesse bloco.
No treino real, o otimizador faz a atualização. Os gradientes acumulam-se por omissão:
é preciso limpá-los antes de calcular os do batch seguinte.

## 3. Obter e preparar o MNIST

Cada exemplo é um par `(imagem, classe)`, com uma classe inteira entre 0 e 9.
`torchvision.datasets.MNIST` implementa a interface **Dataset**: `len(dataset)` dá o número de exemplos
e `dataset[i]` devolve o exemplo de índice `i`.

`ToTensor()` converte a imagem em `float32`, acrescenta a dimensão do canal e transforma a escala
dos píxeis de 0–255 para 0–1. Esta transformação é fixa: não estima estatísticas nos dados.
Se usássemos uma normalização aprendida, estimaríamos os seus parâmetros apenas no treino.

O download é automático na primeira execução e exige Internet. Os ficheiros ficam em
`data/MNIST/raw/`, relativamente ao diretório de trabalho do Jupyter (normalmente a pasta do notebook).
Com `download=True`, os ficheiros válidos já existentes são reutilizados.

In [ ]:
DATA_DIR = Path("data")
transformacao = transforms.ToTensor()
mnist_treino = datasets.MNIST(root=DATA_DIR, train=True, download=True, transform=transformacao)
# O teste é carregado agora, mas só é avaliado depois de fixarmos os modelos.
mnist_teste = datasets.MNIST(root=DATA_DIR, train=False, download=True, transform=transformacao)
print("Pasta dos dados:", DATA_DIR.resolve())
print(f"Conjunto original de treino: {len(mnist_treino):,} | teste: {len(mnist_teste):,}")

### Separar treino, validação e teste

O **treino** ajusta os pesos. A **validação** acompanha a aprendizagem e permite escolher
hiperparâmetros e a epoch a guardar. O **teste** estima o desempenho em dados novos, depois dessas decisões.
Usar repetidamente o teste para melhorar o modelo adapta indiretamente as decisões a esse conjunto.

Para este tutorial usamos **12 000 imagens de treino e 3 000 de validação**, escolhidas aleatoriamente
e sem sobreposição entre as 60 000 imagens originais. As restantes 45 000 ficam sem uso para poupar tempo;
o teste oficial mantém as suas 10 000 imagens. Não estamos a reproduzir um benchmark com todo o treino.
Para usar todas as imagens de desenvolvimento, podes alterar os dois tamanhos para 50 000 e 10 000.

In [ ]:
N_TREINO = 12_000
N_VALIDACAO = 3_000
BATCH_SIZE = 128
EPOCHS = 5
LEARNING_RATE = 1e-3

assert 0 < N_TREINO and 0 < N_VALIDACAO
assert N_TREINO + N_VALIDACAO <= len(mnist_treino)
treino, validacao, nao_utilizado = random_split(
    mnist_treino,
    [N_TREINO, N_VALIDACAO, len(mnist_treino) - N_TREINO - N_VALIDACAO],
    generator=torch.Generator().manual_seed(SEED),
)
assert set(treino.indices).isdisjoint(validacao.indices)
print(f"Treino: {len(treino)} | validação: {len(validacao)} | sem uso: {len(nao_utilizado)}")

fig, axes = plt.subplots(2, 6, figsize=(9, 3.8), constrained_layout=True)
for i, ax in enumerate(axes.flat):
    imagem, classe = treino[i]
    ax.imshow(imagem.squeeze(0), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"Classe: {classe}")
    ax.axis("off")
plt.show()

### De exemplos individuais a mini-batches

Um **DataLoader** agrupa exemplos em batches e, no treino, baralha-os em cada passagem.
Um batch permite calcular uma estimativa do gradiente usando várias imagens em simultâneo.
Uma **epoch** é uma passagem por todos os exemplos do conjunto de treino escolhido.

Usamos `shuffle=True` só no treino. `num_workers=0` simplifica a execução em notebooks,
incluindo Windows. O último batch pode ter menos imagens: não o descartamos.

In [ ]:
def criar_loader(dataset, batch_size=BATCH_SIZE, shuffle=False, seed=SEED):
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0,
        generator=torch.Generator().manual_seed(seed),
    )

train_loader = criar_loader(treino, shuffle=True)
train_eval_loader = criar_loader(treino)
val_loader = criar_loader(validacao)
test_loader = criar_loader(mnist_teste)

X_batch, y_batch = next(iter(train_loader))
print("Imagens:", tuple(X_batch.shape), X_batch.dtype)
print("Classes:", tuple(y_batch.shape), y_batch.dtype)
print(f"Escala dos píxeis: [{X_batch.min().item():.1f}, {X_batch.max().item():.1f}]")
print(f"Batches por epoch: {len(train_loader)} = ceil({len(treino)}/{BATCH_SIZE})")
assert X_batch.shape[1:] == (1, 28, 28) and y_batch.dtype == torch.int64

## 4. Construir uma MLP

Uma camada `nn.Linear(d_in, d_out)` calcula $XW^T+b$. Os pesos $W$ e os biases $b$ são **parâmetros**
aprendidos no treino. A **ReLU**, $\max(0,z)$, introduz não linearidade; sem ativações não lineares,
várias camadas lineares continuariam a representar apenas uma transformação afim.

| Etapa | Forma de saída | Papel |
|---|---|---|
| Entrada | `[B, 1, 28, 28]` | Imagens |
| Flatten | `[B, 784]` | Uma linha por imagem |
| Linear + ReLU | `[B, 128]` | Primeira representação escondida |
| Linear + ReLU | `[B, 64]` | Segunda representação escondida |
| Linear | `[B, 10]` | Um logit por classe |

`nn.Module` regista os parâmetros; `forward` descreve o **forward pass**.
Chamar `modelo(X)` executa esse percurso. `nn.Sequential` aplica as camadas pela ordem indicada.

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden_dims=(128, 64)):
        super().__init__()
        camadas = [nn.Flatten(start_dim=1)]
        entrada = 28 * 28
        for largura in hidden_dims:
            camadas.extend([nn.Linear(entrada, largura), nn.ReLU()])
            entrada = largura
        camadas.append(nn.Linear(entrada, 10))
        self.rede = nn.Sequential(*camadas)

    def forward(self, x):
        return self.rede(x)

def contar_parametros(modelo):
    return sum(p.numel() for p in modelo.parameters() if p.requires_grad)

definir_seed()
mlp = MLP().to(device)
print(mlp)
print(f"Parâmetros treináveis: {contar_parametros(mlp):,}")
with torch.no_grad():
    logits_exemplo = mlp(X_batch.to(device))
print("Forma da saída:", tuple(logits_exemplo.shape))

### Logits, Cross-Entropy e previsão

Os **logits** são valores reais sem restrições: não são probabilidades.
Para uma classe verdadeira $y$, a Cross-Entropy de um exemplo é
$\ell(z,y)=-\log\left(\frac{e^{z_y}}{\sum_{k=0}^{9} e^{z_k}}\right)$.
A loss é menor quando o modelo atribui maior probabilidade à classe correta.

`nn.CrossEntropyLoss()` recebe **logits `[B, 10]`** e **classes inteiras `[B]`** (`torch.long`).
Combina internamente log-softmax e a perda correspondente, de forma numericamente estável.
**Não colocamos softmax na última camada nem antes da loss.**

Para prever uma classe basta `argmax(dim=1)`. Só precisamos de softmax quando queremos visualizar
probabilidades; não altera o índice do maior valor. Essas probabilidades não são garantias de confiança bem calibrada.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
loss_inicial = loss_fn(logits_exemplo, y_batch.to(device))
probabilidades_exemplo = logits_exemplo.softmax(dim=1)
print(f"Loss de um batch antes do treino: {loss_inicial.item():.3f}")
print(f"Referência para probabilidades uniformes: log(10) = {math.log(10):.3f}")
print("Soma das probabilidades da primeira imagem:", probabilidades_exemplo[0].sum().item())
print("Primeiras previsões:", logits_exemplo.argmax(dim=1)[:8].cpu().tolist())
print("Classes verdadeiras:", y_batch[:8].tolist())

Uma loss inicial próxima de `log(10)` é uma referência razoável, não uma igualdade obrigatória:
a inicialização aleatória não produz necessariamente probabilidades uniformes. A accuracy conta acertos;
a loss também depende da probabilidade atribuída à classe verdadeira. Uma pode melhorar sem a outra melhorar.

## 5. Treinar, validar e interpretar

### O que acontece em cada batch?

1. Transferimos imagens e classes para o dispositivo do modelo.
2. `optimizer.zero_grad()` limpa os gradientes anteriores.
3. `modelo(X)` calcula os logits: **forward pass**.
4. A loss compara logits e classes verdadeiras.
5. `loss.backward()` calcula gradientes: **backpropagation**.
6. `optimizer.step()` atualiza os parâmetros.

O otimizador recebe `modelo.parameters()`. Usamos **Adam**, que mantém médias dos gradientes e dos seus
quadrados para adaptar a escala das atualizações de cada parâmetro. O *learning rate* continua a controlar
a escala global dos passos. Em SGD simples, a ideia seria $\theta\leftarrow\theta-\eta\nabla_\theta L$.

`model.train()` ativa o comportamento de treino. `model.eval()` muda o comportamento de camadas como
Dropout e BatchNorm; **não desativa autograd**. Na avaliação usamos também `torch.no_grad()`.
A nossa MLP só tem Linear/ReLU, mas manter esta distinção torna o código reutilizável.

In [ ]:
def treinar_epoch(modelo, loader, optimizer):
    modelo.train()
    soma_loss, acertos, total = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        logits = modelo(X)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()

        n = y.size(0)
        soma_loss += loss.item() * n
        acertos += (logits.argmax(dim=1) == y).sum().item()
        total += n
    return {"loss": soma_loss / total, "accuracy": acertos / total}

Multiplicamos a loss média do batch pelo seu número de exemplos e dividimos pelo total no fim.
Assim, um último batch pequeno não recebe o mesmo peso que um batch completo.
As métricas devolvidas por `treinar_epoch` são medidas **durante** as atualizações:
os primeiros e últimos batches usam pesos diferentes.

Para comparar curvas de treino e validação, vamos medir ambos os conjuntos com os **mesmos pesos no fim
de cada epoch**, em modo de avaliação. Isto acrescenta uma passagem sem gradientes pelo treino.

In [ ]:
def avaliar(modelo, loader):
    modelo.eval()
    soma_loss, acertos, total = 0.0, 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = modelo(X)
            n = y.size(0)
            soma_loss += loss_fn(logits, y).item() * n
            acertos += (logits.argmax(dim=1) == y).sum().item()
            total += n
    return {"loss": soma_loss / total, "accuracy": acertos / total}

### Repetir epochs e conservar o melhor estado

A função seguinte junta as peças já definidas. Guarda uma cópia dos pesos sempre que a **validation loss**
melhora, e restaura o melhor estado no fim. A cópia é necessária: uma referência simples continuaria a
acompanhar as alterações posteriores dos pesos.

O argumento opcional `patience` permite **early stopping**: parar após esse número de epochs seguidas
sem melhoria. Por omissão, executamos as cinco epochs para observar as curvas, mas conservamos sempre
o melhor estado. Não usamos o teste para escolher quando parar.

In [ ]:
def ajustar(modelo, loader_treino, loader_treino_eval, loader_val,
            epochs, learning_rate, patience=None, verbose=True):
    optimizer = torch.optim.Adam(modelo.parameters(), lr=learning_rate)
    historico = []
    melhor_loss = float("inf")
    melhor_estado = None
    melhor_epoch = 0
    sem_melhoria = 0
    inicio = perf_counter()

    for epoch in range(1, epochs + 1):
        treinar_epoch(modelo, loader_treino, optimizer)
        met_treino = avaliar(modelo, loader_treino_eval)
        met_val = avaliar(modelo, loader_val)
        historico.append({
            "epoch": epoch, "train_loss": met_treino["loss"], "val_loss": met_val["loss"],
            "train_acc": met_treino["accuracy"], "val_acc": met_val["accuracy"],
        })
        if met_val["loss"] < melhor_loss:
            melhor_loss = met_val["loss"]
            melhor_estado = deepcopy(modelo.state_dict())
            melhor_epoch = epoch
            sem_melhoria = 0
        else:
            sem_melhoria += 1
        if verbose:
            print(f"Epoch {epoch:2d}/{epochs} | loss treino {met_treino['loss']:.4f} | "
                  f"loss val {met_val['loss']:.4f} | accuracy val {met_val['accuracy']:.2%}")
        if patience is not None and sem_melhoria >= patience:
            print(f"Early stopping na epoch {epoch}.")
            break

    modelo.load_state_dict(melhor_estado)
    return historico, melhor_epoch, perf_counter() - inicio

In [ ]:
# Reinicializamos também o loader para começar com a mesma ordem dos batches.
definir_seed()
mlp = MLP().to(device)
train_loader = criar_loader(treino, shuffle=True)
historico_mlp, melhor_epoch_mlp, tempo_mlp = ajustar(
    mlp, train_loader, train_eval_loader, val_loader, EPOCHS, LEARNING_RATE,
)
print(f"Melhor estado: epoch {melhor_epoch_mlp} | tempo de treino e validação: {tempo_mlp:.1f} s")

In [ ]:
def mostrar_curvas(historico, titulo):
    epochs = [h["epoch"] for h in historico]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), constrained_layout=True)
    for chave, rotulo in [("train", "Treino"), ("val", "Validação")]:
        axes[0].plot(epochs, [h[f"{chave}_loss"] for h in historico], "o-", label=rotulo)
        axes[1].plot(epochs, [100 * h[f"{chave}_acc"] for h in historico], "o-", label=rotulo)
    for ax in axes:
        ax.set_xlabel("Epoch")
        ax.set_xticks(epochs)
        ax.grid(alpha=0.25)
        ax.legend()
    axes[0].set_ylabel("Cross-Entropy média")
    axes[1].set_ylabel("Accuracy (%)")
    fig.suptitle(titulo)
    plt.show()

mostrar_curvas(historico_mlp, "MLP: aprendizagem com os pesos no fim de cada epoch")

**Como ler as curvas.** Se treino e validação melhoram, há evidência de aprendizagem útil.
Se a training loss continua a diminuir enquanto a validation loss aumenta de forma persistente,
há um sinal de **overfitting**: o modelo ajusta detalhes do treino que não ajudam em dados novos.
Um pequeno intervalo entre curvas, por si só, não demonstra overfitting.
Se ambos os desempenhos forem fracos, pode haver capacidade insuficiente, treino insuficiente ou problemas de otimização.

Com poucas epochs, pode não aparecer overfitting visível; não o devemos concluir apenas por o modelo ser uma rede neuronal.
A generalização refere-se ao desempenho em exemplos novos da população de interesse, que o teste apenas estima.

**Experimenta depois:** aumenta `EPOCHS` e usa `patience=2` na chamada de `ajustar`.
O estado restaurado pode pertencer a uma epoch anterior à última representada no gráfico.

## 6. Pequenas experiências com hiperparâmetros

Os pesos são aprendidos; os **hiperparâmetros** são escolhas do processo: learning rate, batch size,
número de epochs, número de camadas e respetivas larguras.

Vamos alterar **um fator de cada vez** em cinco experiências com apenas 4 000 imagens de treino e duas epochs.
Todas usam a mesma amostra e validação; cada uma começa com um modelo e um otimizador novos.
As três primeiras têm a mesma inicialização e ordem de batches, isolando melhor o efeito do learning rate.
Comparar larguras muda também a inicialização e a capacidade.

Estas experiências curtas desenvolvem intuição; não selecionam automaticamente a configuração do modelo principal,
que fixámos antecipadamente. O desempenho aqui não é diretamente comparável ao da MLP treinada com mais dados e epochs.

In [ ]:
treino_exp = Subset(treino, range(min(4_000, len(treino))))
exp_eval_loader = criar_loader(treino_exp)
EPOCHS_EXP = 2
configuracoes = [
    {"nome": "Referência", "lr": 1e-3, "batch": 128, "hidden": (128, 64)},
    {"nome": "LR menor", "lr": 1e-4, "batch": 128, "hidden": (128, 64)},
    {"nome": "LR maior", "lr": 1e-2, "batch": 128, "hidden": (128, 64)},
    {"nome": "Batch maior", "lr": 1e-3, "batch": 256, "hidden": (128, 64)},
    {"nome": "Menos neurónios", "lr": 1e-3, "batch": 128, "hidden": (64, 32)},
]

resultados_exp = []
for cfg in configuracoes:
    definir_seed()
    modelo_exp = MLP(hidden_dims=cfg["hidden"]).to(device)
    loader_exp = criar_loader(treino_exp, batch_size=cfg["batch"], shuffle=True)
    hist, ep, _ = ajustar(modelo_exp, loader_exp, exp_eval_loader, val_loader,
                         EPOCHS_EXP, cfg["lr"], verbose=False)
    metricas = avaliar(modelo_exp, val_loader)
    resultados_exp.append({"config": cfg, "historico": hist, "metricas": metricas, "epoch": ep})

print(f"{'Experiência':<19} {'LR':>8} {'Batch':>7} {'Camadas':>12} {'Epoch*':>7} {'Val loss':>10} {'Val acc':>9}")
for r in resultados_exp:
    c, m = r["config"], r["metricas"]
    print(f"{c['nome']:<19} {c['lr']:8.4f} {c['batch']:7d} {str(c['hidden']):>12} "
          f"{r['epoch']:7d} {m['loss']:10.4f} {m['accuracy']:8.2%}")
print("* Estado escolhido pela menor validation loss, dentro de cada experiência.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5), constrained_layout=True)
for r in resultados_exp[:3]:
    ax.plot([h["epoch"] for h in r["historico"]],
            [h["val_loss"] for h in r["historico"]], "o-", label=r["config"]["nome"])
ax.set(xlabel="Epoch", ylabel="Validation loss", title="Learning rate: mesmo orçamento de duas epochs")
ax.set_xticks(range(1, EPOCHS_EXP + 1))
ax.legend()
ax.grid(alpha=0.25)
plt.show()

**Interpreta os resultados obtidos:**

- Um learning rate pequeno pode aprender lentamente; um grande pode progredir depressa ou desestabilizar o treino.
  A ordenação após duas epochs pode mudar com um orçamento maior.
- Um batch maior dá menos atualizações por epoch. Aqui mantivemos o número de passagens pelos dados,
  não o número de passos do otimizador.
- Mais neurónios aumentam o número de parâmetros, mas não garantem melhor generalização.
- O número de epochs também é um hiperparâmetro: as curvas anteriores mostram o efeito de prolongar o treino.

**Explora:** acrescenta uma configuração com `hidden=(128,)`, mantendo os restantes valores da referência.
Depois repete com outra seed. Uma única execução não mede a variabilidade dos resultados.
Se quisesses escolher efetivamente um hiperparâmetro, usarias a validação, confirmarias com um orçamento
adequado e só depois avaliarias a configuração final no teste.

## 7. Extensão: uma CNN pequena

A MLP pode aprender relações entre píxeis, mas não tem uma preferência explícita por padrões locais.
Uma **convolução** aplica os mesmos filtros a várias posições: explora vizinhanças e partilha pesos.
Isto introduz uma estrutura apropriada para imagens. Não garante invariância a todas as transformações.
O pooling reduz a resolução espacial e a quantidade de cálculo.

Vamos manter as imagens em `[B, 1, 28, 28]` até às convoluções e só achatar no classificador.
`padding=1` numa convolução 3×3 com stride 1 mantém a altura e largura.

| Operação | Forma de saída |
|---|---|
| Conv 1→8, 3×3 + ReLU | `[B, 8, 28, 28]` |
| MaxPool 2×2 | `[B, 8, 14, 14]` |
| Conv 8→16, 3×3 + ReLU | `[B, 16, 14, 14]` |
| MaxPool 2×2 | `[B, 16, 7, 7]` |
| Flatten → Linear → ReLU → Linear | `[B, 10]` |

Reutilizamos a loss, os loaders e o training loop. Fixamos a arquitetura antes do teste e usamos o mesmo
treino, validação, learning rate, batch size e número máximo de epochs da MLP. Tempos e parâmetros podem diferir.

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classificador = nn.Sequential(
            nn.Flatten(), nn.Linear(16 * 7 * 7, 64), nn.ReLU(), nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.classificador(self.features(x))

definir_seed()
cnn = CNN().to(device)
with torch.no_grad():
    assert cnn(X_batch.to(device)).shape == (len(X_batch), 10)
print(f"Parâmetros: MLP {contar_parametros(mlp):,} | CNN {contar_parametros(cnn):,}")
historico_cnn, melhor_epoch_cnn, tempo_cnn = ajustar(
    cnn, criar_loader(treino, shuffle=True), train_eval_loader, val_loader,
    EPOCHS, LEARNING_RATE,
)
mostrar_curvas(historico_cnn, "CNN: o mesmo processo de treino")

## 8. Avaliação final e análise de previsões

Agora congelamos as decisões: MLP e CNN têm os estados de menor validation loss das respetivas execuções.
Se fosse necessário escolher uma delas, faríamos essa escolha **pela validação**, antes de olhar para o teste.
Vamos reportar ambos os modelos como comparação didática previamente definida.

A accuracy compara o número de classes corretamente previstas com o total. Como as classes MNIST têm
frequências relativamente próximas, é uma primeira medida útil. Noutros problemas pode ocultar mau
desempenho nas classes raras; a balanced accuracy, média dos recalls por classe, dá o mesmo peso a cada classe.

In [ ]:
modelos = {"MLP": mlp, "CNN": cnn}
val_finais = {nome: avaliar(modelo, val_loader) for nome, modelo in modelos.items()}
escolha_validacao = min(val_finais, key=lambda nome: val_finais[nome]["loss"])
print("Escolha pela validation loss (antes de avaliar o teste):", escolha_validacao)

# Uma passagem por modelo no teste; guardamos as previsões para toda a análise seguinte.
resultados_teste = {}
for nome, modelo in modelos.items():
    modelo.eval()
    soma_loss = 0.0
    probabilidades, verdadeiros = [], []
    with torch.no_grad():
        for X, y in test_loader:
            logits = modelo(X.to(device))
            soma_loss += loss_fn(logits, y.to(device)).item() * y.size(0)
            probabilidades.append(logits.softmax(dim=1).cpu())
            verdadeiros.append(y)
    probs = torch.cat(probabilidades)
    y_real = torch.cat(verdadeiros)
    y_pred = probs.argmax(dim=1)
    matriz = torch.bincount(10 * y_real + y_pred, minlength=100).reshape(10, 10)
    recall_classes = matriz.diag().float() / matriz.sum(dim=1).clamp_min(1)
    resultados_teste[nome] = {
        "loss": soma_loss / len(y_real), "accuracy": (y_pred == y_real).float().mean().item(),
        "balanced_accuracy": recall_classes.mean().item(), "matriz": matriz,
        "probs": probs, "y_real": y_real, "y_pred": y_pred,
    }

print(f"{'Modelo':<8} {'Parâmetros':>11} {'Tempo (s)':>10} {'Val loss':>10} {'Test loss':>10} {'Test acc':>10} {'Bal. acc':>10}")
for nome, tempo in [("MLP", tempo_mlp), ("CNN", tempo_cnn)]:
    r = resultados_teste[nome]
    print(f"{nome:<8} {contar_parametros(modelos[nome]):11,d} {tempo:10.1f} "
          f"{val_finais[nome]['loss']:10.4f} {r['loss']:10.4f} {r['accuracy']:10.2%} {r['balanced_accuracy']:10.2%}")

O tempo inclui treino, medição de métricas e cópia dos melhores pesos; não inclui download nem as experiências
da secção 6. Não é uma comparação de velocidade otimizada entre arquiteturas.
Uma CNN pode obter melhor accuracy com menos parâmetros, mas a conclusão deve apoiar-se na tabela obtida.
Mesmo com o mesmo número de epochs, os modelos têm custos e trajetórias de otimização diferentes.
Uma execução com uma seed e uma amostra de dados não estabelece superioridade geral.

Após consultar estes resultados, alterar configurações para melhorar o teste tornaria esta avaliação parte
do processo de seleção. Para continuar a experimentar, usa a validação e reserva novos dados para uma avaliação independente.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
for ax, nome in zip(axes, modelos):
    matriz = resultados_teste[nome]["matriz"]
    mapa = ax.imshow(matriz, cmap="Blues")
    ax.set(title=f"{nome}: matriz de confusão (contagens)", xlabel="Classe prevista", ylabel="Classe verdadeira")
    ax.set_xticks(range(10))
    ax.set_yticks(range(10))
    fig.colorbar(mapa, ax=ax, shrink=0.8)
plt.show()

Na matriz de confusão, a diagonal contém os acertos. Uma célula fora da diagonal na linha 4 e coluna 9
conta dígitos 4 previstos como 9. As cores representam contagens; cada figura tem a sua escala.
Vamos observar também acertos e erros da MLP. Usamos as previsões já guardadas, sem voltar a avaliar o teste.

In [ ]:
r = resultados_teste["MLP"]
corretos = (r["y_pred"] == r["y_real"]).nonzero(as_tuple=True)[0]
incorretos = (r["y_pred"] != r["y_real"]).nonzero(as_tuple=True)[0]
fig, axes = plt.subplots(2, 6, figsize=(11, 4.6), constrained_layout=True)
for linha, (indices, titulo) in enumerate([(corretos, "Acerto"), (incorretos, "Erro")]):
    for coluna, ax in enumerate(axes[linha]):
        ax.axis("off")
        if coluna >= len(indices):
            ax.set_title("Sem exemplo disponível")
            continue
        idx = indices[coluna].item()
        imagem, classe = mnist_teste[idx]
        pred = r["y_pred"][idx].item()
        prob = r["probs"][idx, pred].item()
        ax.imshow(imagem.squeeze(0), cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"{titulo}: real {classe} → {pred}\np(classe prevista)={prob:.2f}", fontsize=9)
plt.show()
print(f"MLP: {len(corretos)} acertos e {len(incorretos)} erros no teste.")

**Para refletir.** Os erros parecem visualmente ambíguos? Há previsões erradas com probabilidade elevada?
Uma probabilidade elevada não prova que a previsão está correta. Estes são os primeiros acertos e erros na
ordem do teste, não uma amostra aleatória representativa de todos os casos difíceis.

## 9. Guardar e reutilizar o modelo

Um `state_dict` contém os parâmetros e outros estados registados do modelo.
Guardamos os pesos da MLP selecionados pela validação e a configuração mínima para reconstruir a arquitetura.
Para inferência, é preciso aplicar **a mesma transformação de entrada** e usar `eval()`.
Este ficheiro permite reutilizar o modelo; retomar exatamente o treino exigiria também o estado do otimizador e das seeds.

Os artefactos gerados ficam em `artefactos/`, junto da pasta `data/`, no diretório de trabalho.

In [ ]:
ARTEFACTOS_DIR = Path("artefactos")
ARTEFACTOS_DIR.mkdir(exist_ok=True)
checkpoint_path = ARTEFACTOS_DIR / "mlp_mnist.pt"
torch.save({
    "state_dict": {nome: tensor.detach().cpu() for nome, tensor in mlp.state_dict().items()},
    "hidden_dims": [128, 64],
    "input_shape": [1, 28, 28],
    "classes": list(range(10)),
    "transformacao": "ToTensor: float32, um canal, escala 0–1",
    "melhor_epoch": melhor_epoch_mlp,
}, checkpoint_path)

# Carregamos o ficheiro que acabámos de criar.
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
mlp_recarregada = MLP(hidden_dims=checkpoint["hidden_dims"]).to(device)
mlp_recarregada.load_state_dict(checkpoint["state_dict"])
mlp_recarregada.eval()
mlp.eval()
with torch.no_grad():
    entrada = X_batch[:8].to(device)
    torch.testing.assert_close(mlp(entrada), mlp_recarregada(entrada))
    previsoes = mlp_recarregada(entrada).argmax(dim=1).cpu().tolist()
print("Modelo guardado em:", checkpoint_path)
print("Previsões após carregar:", previsoes)
print("Verificação concluída: os logits coincidem com os do modelo original.")

## 10. O que levar para o projeto

O percurso `Dataset → DataLoader → nn.Module → loss → backward → optimizer → avaliação` mantém-se.
A representação dos dados, a dimensão de entrada e o número de classes devem corresponder ao problema.
Uma MLP recebe vetores de atributos; a CNN deste tutorial recebe imagens de dígitos e não é diretamente
uma arquitetura para sequências de poses.

No projeto existem várias sequências por paciente. Uma divisão aleatória por sequência pode colocar o mesmo
paciente no treino e na validação, dando uma estimativa demasiado otimista da generalização a novos pacientes.
A separação de pacientes de treino e teste definida no enunciado deve ser preservada; a validação para escolher
modelos deve também respeitar os grupos de pacientes dentro do conjunto de desenvolvimento.
Qualquer normalização ou representação aprendida deve ser ajustada apenas nos dados de treino de cada divisão.

Antes de desenvolver o teu modelo, confirma que consegues explicar:

1. Por que tem a MLP 784 entradas e 10 saídas, mas a loss recebe apenas um inteiro por imagem?
2. O que distingue `backward()` de `optimizer.step()`? O que aconteceria sem `zero_grad()`?
3. Por que são necessários `eval()` **e** `no_grad()` na avaliação?
4. Por que guardamos o estado de menor validation loss em vez de escolher pelo teste?
5. Que padrão nas curvas sugeriria overfitting? Que outras explicações podem existir para desempenho fraco?

**Respostas breves para autoavaliação.** (1) 784 píxeis; 10 logits; o inteiro identifica a classe verdadeira.
(2) `backward` calcula gradientes e `step` altera parâmetros; sem limpeza, os gradientes acumulam-se.
(3) Um muda o comportamento de certas camadas, o outro desativa o registo para diferenciação.
(4) A validação orienta decisões; o teste deve avaliar as decisões já fixadas.
(5) Treino a melhorar com deterioração persistente na validação; desempenho fraco em ambos pode dever-se
a otimização, treino insuficiente, representação inadequada ou capacidade insuficiente.

### Materiais e referências

- **IAAut_Aprendizagem_Supervisionada.pdf**, introdução e Parte V, pp. 12–14: dependências, redes neuronais,
  seleção de modelos e generalização entre pacientes. O exemplo MNIST é um apoio à aprendizagem destes conceitos.
- [PyTorch: otimizar os parâmetros](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)
  — training loop, gradientes e modos de execução.
- [PyTorch: CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)
  — contrato de logits e classes.
- [torchvision: MNIST](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.MNIST.html)
  — carregamento, transformação e reutilização dos ficheiros descarregados.